In [4]:
# ====== Setup ======
import math
import re
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

np.random.seed(42)

# ====== Scoring ======
@dataclass
class TaskResult:
    passed: bool
    points_awarded: int
    points_max: int
    message: str

_RESULTS: Dict[str, TaskResult] = {}

def _ok(task: str, pts: int, pts_max: int, msg: str = "OK"):
    _RESULTS[task] = TaskResult(True, pts, pts_max, msg)

def _fail(task: str, pts_max: int, msg: str):
    _RESULTS[task] = TaskResult(False, 0, pts_max, msg)

def _require(condition: bool, msg: str):
    if not condition:
        raise AssertionError(msg)

def summary():
    total = sum(r.points_awarded for r in _RESULTS.values())
    total_max = sum(r.points_max for r in _RESULTS.values())
    print("=== SCORE SUMMARY ===")
    for k, r in _RESULTS.items():
        status = "PASS" if r.passed else "FAIL"
        print(f"{k:>6} | {status:4} | {r.points_awarded:3}/{r.points_max:3} | {r.message}")
    print(f"\nTOTAL: {total}/{total_max}")
    if total < 30:
        print("LEVEL: A (start od podstaw)")
    elif total < 60:
        print("LEVEL: B (standardowy)")
    elif total < 85:
        print("LEVEL: C (CV, strojenie, analiza błędów)")
    else:
        print("LEVEL: D (projekty i rozszerzenia)")

In [5]:
# TASK 1 (10 pts)
def clean_text(s: str) -> str:
    """
    TODO:
    - zamień na lowercase
    - usuń znaki niebędące literą/cyfrą/spacją (czyli interpunkcję)
    - zredukuj wielokrotne spacje do jednej
    - obetnij spacje z początku/końca
    """
    # --- TODO ---
    raise NotImplementedError

In [6]:
def test_task1():
    task = "T1"
    pts = 10
    try:
        _require(clean_text(" Hello,  WORLD!! ") == "hello world", "clean_text basic failed")
        _require(clean_text("Ala-ma_kota") == "ala ma kota", "clean_text punctuation failed")
        _require(clean_text("  x   y   z  ") == "x y z", "clean_text whitespace failed")
        _ok(task, pts, pts)
    except Exception as e:
        _fail(task, pts, str(e))

test_task1()
summary()

=== SCORE SUMMARY ===
    T1 | FAIL |   0/ 10 | 

TOTAL: 0/10
LEVEL: A (start od podstaw)


In [7]:
# TASK 2 (10 pts)
def moving_average(x: np.ndarray, window: int) -> np.ndarray:
    """
    TODO:
    - x: 1D array
    - zwróć średnią kroczącą o długości 'window'
    - wynik ma mieć długość len(x)-window+1
    - użyj NumPy (preferowane: konwolucja / cumsum), bez pętli for
    """
    # --- TODO ---
    raise NotImplementedError

In [8]:
def test_task2():
    task = "T2"
    pts = 10
    try:
        x = np.array([1,2,3,4,5], dtype=float)
        y = moving_average(x, 2)
        _require(np.allclose(y, np.array([1.5,2.5,3.5,4.5])), "moving_average window=2 failed")
        y = moving_average(x, 5)
        _require(np.allclose(y, np.array([3.0])), "moving_average window=5 failed")
        _ok(task, pts, pts)
    except Exception as e:
        _fail(task, pts, str(e))

test_task2()
summary()

=== SCORE SUMMARY ===
    T1 | FAIL |   0/ 10 | 
    T2 | FAIL |   0/ 10 | 

TOTAL: 0/20
LEVEL: A (start od podstaw)


In [9]:
# TASK 3 (15 pts)
def summarize_by_group(df: pd.DataFrame, group_col: str, value_col: str) -> pd.DataFrame:
    """
    TODO:
    - zwróć DataFrame z kolumnami:
      [group_col, "count", "mean", "std"]
    - posortuj po group_col rosnąco
    """
    # --- TODO ---
    raise NotImplementedError

In [10]:
def test_task3():
    task = "T3"
    pts = 15
    try:
        df = pd.DataFrame({
            "group": ["A","A","B","B","B"],
            "val":   [1,  3,  2,  2,  4]
        })
        out = summarize_by_group(df, "group", "val")
        _require(list(out.columns) == ["group","count","mean","std"], "bad columns")
        _require(out.loc[out["group"]=="A","count"].iloc[0] == 2, "count A wrong")
        _require(math.isclose(out.loc[out["group"]=="A","mean"].iloc[0], 2.0), "mean A wrong")
        _require(out.shape[0] == 2, "rows wrong")
        _ok(task, pts, pts)
    except Exception as e:
        _fail(task, pts, str(e))

test_task3()
summary()

=== SCORE SUMMARY ===
    T1 | FAIL |   0/ 10 | 
    T2 | FAIL |   0/ 10 | 
    T3 | FAIL |   0/ 15 | 

TOTAL: 0/35
LEVEL: A (start od podstaw)


In [11]:
# TASK 4 (10 pts)
def merge_and_fill(users: pd.DataFrame, scores: pd.DataFrame) -> pd.DataFrame:
    """
    TODO:
    - users: columns ["user_id","name"]
    - scores: columns ["user_id","score"]
    - zrób left join (wszyscy users mają zostać)
    - brakujące score uzupełnij 0
    - wynik ma mieć kolumny ["user_id","name","score"] i być posortowany po user_id
    """
    # --- TODO ---
    raise NotImplementedError

In [12]:
def test_task4():
    task = "T4"
    pts = 10
    try:
        users = pd.DataFrame({"user_id":[2,1,3], "name":["B","A","C"]})
        scores = pd.DataFrame({"user_id":[1,3], "score":[10, 5]})
        out = merge_and_fill(users, scores)
        _require(list(out.columns) == ["user_id","name","score"], "bad columns")
        _require(out["user_id"].tolist() == [1,2,3], "not sorted")
        _require(out.loc[out["user_id"]==2,"score"].iloc[0] == 0, "fillna wrong")
        _ok(task, pts, pts)
    except Exception as e:
        _fail(task, pts, str(e))

test_task4()
summary()

=== SCORE SUMMARY ===
    T1 | FAIL |   0/ 10 | 
    T2 | FAIL |   0/ 10 | 
    T3 | FAIL |   0/ 15 | 
    T4 | FAIL |   0/ 10 | 

TOTAL: 0/45
LEVEL: A (start od podstaw)


In [13]:
# TASK 5 (25 pts)
def baseline_vs_model(X: np.ndarray, y: np.ndarray, test_size: float = 0.25, seed: int = 42) -> Dict[str, float]:
    """
    TODO:
    - wykonaj stratified train/test split
    - baseline: DummyClassifier(most_frequent)
    - model: Pipeline(StandardScaler + LogisticRegression(max_iter=200))
    - zwróć dict:
      {"baseline_acc": ..., "model_acc": ..., "model_f1": ...}
    """
    # --- TODO ---
    raise NotImplementedError

In [14]:
def test_task5():
    task = "T5"
    pts = 25
    try:
        rng = np.random.default_rng(42)
        n = 400
        X = rng.normal(size=(n, 2))
        # prosta granica: y = 1, jeśli x0 + 0.5*x1 > 0
        y = (X[:,0] + 0.5*X[:,1] > 0).astype(int)
        out = baseline_vs_model(X, y)

        _require(set(out.keys()) == {"baseline_acc","model_acc","model_f1"}, "bad keys")
        _require(0.0 <= out["baseline_acc"] <= 1.0, "baseline_acc out of range")
        _require(0.0 <= out["model_acc"] <= 1.0, "model_acc out of range")
        _require(0.0 <= out["model_f1"] <= 1.0, "model_f1 out of range")
        _require(out["model_acc"] > out["baseline_acc"], "model should beat baseline here")
        _ok(task, pts, pts)
    except Exception as e:
        _fail(task, pts, str(e))

test_task5()
summary()

=== SCORE SUMMARY ===
    T1 | FAIL |   0/ 10 | 
    T2 | FAIL |   0/ 10 | 
    T3 | FAIL |   0/ 15 | 
    T4 | FAIL |   0/ 10 | 
    T5 | FAIL |   0/ 25 | 

TOTAL: 0/70
LEVEL: A (start od podstaw)


In [15]:
# TASK 6 (30 pts)
def tune_knn_cv(X: np.ndarray, y: np.ndarray, k_values: List[int], seed: int = 42) -> Tuple[int, Dict[int, float]]:
    """
    TODO:
    - dla każdego k z k_values zbuduj Pipeline(StandardScaler + KNN(k))
    - policz mean accuracy w StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    - zwróć:
      (best_k, {k: mean_cv_acc})
    - best_k: największa mean accuracy; przy remisie wybierz mniejsze k
    """
    # --- TODO ---
    raise NotImplementedError

In [16]:
def test_task6():
    task = "T6"
    pts = 30
    try:
        rng = np.random.default_rng(1)
        n = 300
        X = rng.normal(size=(n, 3))
        y = (X[:,0] - X[:,1] + 0.2*rng.normal(size=n) > 0).astype(int)

        best_k, scores = tune_knn_cv(X, y, k_values=[1,3,5,7,9])
        _require(best_k in [1,3,5,7,9], "best_k not in k_values")
        _require(isinstance(scores, dict) and all(k in scores for k in [1,3,5,7,9]), "scores missing keys")
        _require(all(0 <= v <= 1 for v in scores.values()), "scores out of range")
        _ok(task, pts, pts)
    except Exception as e:
        _fail(task, pts, str(e))

test_task6()
summary()

=== SCORE SUMMARY ===
    T1 | FAIL |   0/ 10 | 
    T2 | FAIL |   0/ 10 | 
    T3 | FAIL |   0/ 15 | 
    T4 | FAIL |   0/ 10 | 
    T5 | FAIL |   0/ 25 | 
    T6 | FAIL |   0/ 30 | 

TOTAL: 0/100
LEVEL: A (start od podstaw)


In [17]:
def run_all_tests():
    _RESULTS.clear()
    test_task1()
    test_task2()
    test_task3()
    test_task4()
    test_task5()
    test_task6()
    summary()

run_all_tests()

=== SCORE SUMMARY ===
    T1 | FAIL |   0/ 10 | 
    T2 | FAIL |   0/ 10 | 
    T3 | FAIL |   0/ 15 | 
    T4 | FAIL |   0/ 10 | 
    T5 | FAIL |   0/ 25 | 
    T6 | FAIL |   0/ 30 | 

TOTAL: 0/100
LEVEL: A (start od podstaw)
